In [1]:
# Install spaCy (if not already installed, Colab sometimes has it)
!pip install -U spacy

# Download the small English language model
# The -q flag keeps the output quiet and clean
!python -m spacy download en_core_web_sm -q

print("spaCy and model installed successfully!")

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 12.8/12.8 MB 101.6 MB/s eta 0:00:00
✔ Download and installation successful
You can now load the package via spacy.load('en_core_web_sm')
⚠ Restart to reload dependencies
If you are in a Jupyter or Colab notebook, you may need to restart Python in
order to load all the package's dependencies. You can do this by selecting the
'Restart kernel' or 'Restart runtime' option.
spaCy and model installed successfully!


In [2]:
import spacy
from spacy import displacy # Used for the nice visualization

# Load the small English model
# 'nlp' (Natural Language Processor) is the standard object name
nlp = spacy.load("en_core_web_sm")

print("spaCy model loaded and ready.")

spaCy model loaded and ready.


In [3]:
# Our sample text (contains PERSON, ORGANIZATION, LOCATION, and MONEY entities)
text = (
    "Tim Cook, the CEO of Apple, announced the new iPhone in Cupertino, "
    "California, last Tuesday. The deal is worth $1,000,000."
)

# Process the text with the model
doc = nlp(text)

print(f"Processed Text: \n'{text}'\n")
print(f"Number of Entities Found: {len(doc.ents)}")

Processed Text: 
'Tim Cook, the CEO of Apple, announced the new iPhone in Cupertino, California, last Tuesday. The deal is worth $1,000,000.'

Number of Entities Found: 6


In [4]:
print("--- Named Entities Extracted ---")
print(f"{'Entity Text':<20} | {'Entity Type':<15} | Explanation")
print("-" * 55)

# Iterate through all entities in the document
for ent in doc.ents:
    # Use spacy.explain() to get the human-readable description of the label
    explanation = spacy.explain(ent.label_)

    print(f"{ent.text:<20} | {ent.label_:<15} | {explanation}")

--- Named Entities Extracted ---
Entity Text          | Entity Type     | Explanation
-------------------------------------------------------
Tim Cook             | PERSON          | People, including fictional
Apple                | ORG             | Companies, agencies, institutions, etc.
Cupertino            | GPE             | Countries, cities, states
California           | GPE             | Countries, cities, states
last Tuesday         | DATE            | Absolute or relative dates or periods
1,000,000            | MONEY           | Monetary values, including unit


In [5]:
# Render the processed document with the 'ent' (entities) style.
# jupyter=True is necessary for correct rendering in Colab.
displacy.render(doc, style="ent", jupyter=True)

# You can also customize the entity types displayed
# Only show Person (PERSON) and Organization (ORG) entities
options = {"ents": ["PERSON", "ORG"], "colors": {"PERSON": "#ffcc5c", "ORG": "#aa9cfc"}}
displacy.render(doc, style="ent", jupyter=True, options=options)

In [6]:
# Import the EntityRuler
from spacy.pipeline import EntityRuler

# Create a blank English model to add our custom rules to.
# We will use this new model instead of the default 'en_core_web_sm'.
custom_nlp = spacy.blank("en")

# Create the EntityRuler component
ruler = EntityRuler(custom_nlp)

# Define your custom patterns and their labels
# 'Lesser-Known Tech Product' and 'Tech Buzzword' are custom entity types
patterns = [
    {"label": "TECH_PRODUCT", "pattern": "NanoChip X3"},
    {"label": "TECH_PRODUCT", "pattern": "Quantum OS"},
    {"label": "BUZZWORD", "pattern": "DeFi Protocol"},
    {"label": "PERSON", "pattern": "Jane Doe"}
]

# Add the patterns to the ruler
ruler.add_patterns(patterns)

# Add the ruler to the pipeline of our custom model
custom_nlp.add_pipe("entity_ruler").add_patterns(patterns)

print(f"Custom EntityRuler loaded with {len(patterns)} rules.")

Custom EntityRuler loaded with 4 rules.


In [7]:
# A new sample text to test the custom rules
custom_text = (
    "The new **NanoChip X3** is driving the **DeFi Protocol** forward. "
    "Our lead engineer, **Jane Doe**, showcased the **Quantum OS** upgrade last week."
)

# Process the new text with the custom model
custom_doc = custom_nlp(custom_text)

print("--- Custom Entities Found ---")
for ent in custom_doc.ents:
    print(f"{ent.text:<20} | {ent.label_}")

# Visualize the custom entities
# Use a custom set of colors for your new labels for a better presentation
custom_colors = {
    "TECH_PRODUCT": "linear-gradient(90deg, #aa9cfc, #fc9ce7)",
    "BUZZWORD": "lightgreen",
    "PERSON": "yellow"
}
options = {"colors": custom_colors}

displacy.render(custom_doc, style="ent", jupyter=True, options=options)

--- Custom Entities Found ---
NanoChip X3          | TECH_PRODUCT
DeFi Protocol        | BUZZWORD
Jane Doe             | PERSON
Quantum OS           | TECH_PRODUCT
